In [30]:
import numpy as np
import joblib

from tensorflow.keras.models import load_model
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve

In [31]:
# Load trained models (NO TRAINING HERE)
autoencoder = load_model("../models/autoencoder.h5",compile=False)
iso_forest = joblib.load("../models/isolation_forest.pkl")

print("Models loaded successfully")

Models loaded successfully


In [32]:
X_test_scaled = np.load("X_test_scaled.npy")
y_test = np.load("y_test.npy")

print("Test data shape:", X_test_scaled.shape)

Test data shape: (25192, 116)


In [33]:
# Reconstruct test data
reconstructions = autoencoder.predict(X_test_scaled)

# Compute MSE (reconstruction error)
mse = np.mean((X_test_scaled - reconstructions) ** 2, axis=1)

print("Reconstruction error computed")

788/788 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Reconstruction error computed


In [34]:
# Get anomaly scores (higher = more anomalous)
if_scores = iso_forest.decision_function(X_test_scaled)

# Convert to anomaly score
if_scores = -if_scores

print("Isolation Forest scores computed")

Isolation Forest scores computed


In [35]:
ae_scaler = MinMaxScaler()
if_scaler = MinMaxScaler()

ae_norm = ae_scaler.fit_transform(mse.reshape(-1, 1)).flatten()
if_norm = if_scaler.fit_transform(if_scores.reshape(-1, 1)).flatten()

print("Scores normalized")

Scores normalized


In [36]:
alpha = 0.6  # weight for autoencoder

hybrid_score = alpha * ae_norm + (1 - alpha) * if_norm

print("Hybrid score calculated")

Hybrid score calculated


In [37]:
# Use ROC curve to find best threshold
fpr, tpr, thresholds = roc_curve(y_test, hybrid_score)

optimal_idx = np.argmax(tpr - fpr)
threshold = thresholds[optimal_idx]

print("Optimal threshold:", threshold)

Optimal threshold: 0.12792192938505223


In [38]:
y_pred_hybrid = (hybrid_score > threshold).astype(int)

print("Predictions generated")

Predictions generated


In [39]:
print("\n=== Classification Report ===")
print(classification_report(y_test, y_pred_hybrid))

print("\n=== Confusion Matrix ===")
print(confusion_matrix(y_test, y_pred_hybrid))


=== Classification Report ===
              precision    recall  f1-score   support

           0       0.89      0.82      0.85     13449
           1       0.81      0.88      0.85     11743

    accuracy                           0.85     25192
   macro avg       0.85      0.85      0.85     25192
weighted avg       0.85      0.85      0.85     25192


=== Confusion Matrix ===
[[11064  2385]
 [ 1403 10340]]


In [40]:
tn, fp, fn, tp = confusion_matrix(y_test, y_pred_hybrid).ravel()

fpr_value = fp / (fp + tn)

print("False Positive Rate:", fpr_value)

False Positive Rate: 0.17733660495204104


In [41]:
np.save("hybrid_scores.npy", hybrid_score)
np.save("y_pred_hybrid.npy", y_pred_hybrid)

print("Hybrid outputs saved")

Hybrid outputs saved
